In [112]:
import os
import MeCab
from langchain_community.document_loaders import JSONLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

def extract_metadata(record: dict, metadata: dict) -> dict:
    # 'record' is the parsed JSON object
    # If the JSON contains a 'metadata' key, extract its fields
    if "metadata" in record and isinstance(record["metadata"], dict):
        source_metadata = record["metadata"]
        metadata["title_ja"] = source_metadata.get("title_ja")
        metadata["title_en"] = source_metadata.get("title_en")
        metadata["year"] = source_metadata.get("year")
        metadata["characters"] = source_metadata.get("characters", [])
        metadata["mediatype"] = source_metadata.get("mediatype")
        metadata["author"] = source_metadata.get("author")
        metadata["url"] = source_metadata.get("url")
    return metadata

def load_documents(docs_path="docs"):
    print(f"Loading documents from {docs_path}...")

    if not os.path.exists(docs_path):
        raise FileNotFoundError(f"The directory {docs_path} does not exist.")

    # Load all .txt files from the docs directory
    loader = DirectoryLoader(
        path=docs_path,
        glob="*.json",
        loader_cls=JSONLoader,
        loader_kwargs={
            "jq_schema": ".",                 # Get the whole JSON object
            "content_key": "page_content",    # Pick the text key
            "metadata_func": extract_metadata,
            "text_content": False
        }
    )
    documents = loader.load()
    if len(documents) == 0:
        raise FileNotFoundError(f"No .json files found in {docs_path}.")
    # Add the media_type and clean the source metadat
    return documents



# Initialize the Tagger
# unidic-lite is automatically detected by mecab-python3
tagger = MeCab.Tagger()

def mecab_len(text):
    """Counts the number of tokens in the text using MeCab."""
    node = tagger.parseToNode(text)
    count = 0
    while node:
        if node.surface:
            count += 1
        node = node.next
    return count

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,           # Limit chunks to 100 MeCab tokens
    chunk_overlap=150,         # 20 token overlap
    length_function=mecab_len, # Use our custom MeCab counter
    separators=["\n\n", "\n", "。", "、", " ", ""] # Japanese-friendly separators
)

def split_documents(documents):
    chunks = text_splitter.split_documents(documents)
    title_counters = {}

    for chunk in chunks:
        source_title = chunk.metadata.get("title_en", "unknown_source")

        if source_title not in title_counters:
            title_counters[source_title] = 0
        else:
            title_counters[source_title] += 1

        # Create a structured, predictable ID
        raw_tracking_id = f"{source_title}_chunk_{title_counters[source_title]}"
        ascii_safe_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, raw_tracking_id))

        # Inject the ID into the chunk's native metadata dictionary
        chunk.metadata["id"] = ascii_safe_id

    return chunks

def deduplicate_by_filename(docs):
    unique_docs = {}

    for doc in docs:
        filename = os.path.basename(doc.metadata["source"])
        current_type = doc.metadata.get("mediatype", "unknown")

        if filename not in unique_docs:
            # First time seeing this file: store the doc
            unique_docs[filename] = doc
            # Convert the single string into a set of types
            unique_docs[filename].metadata["mediatype"] = {current_type}
        else:
            # Already exists: add the new media_type to the set
            unique_docs[filename].metadata["mediatype"].add(current_type)

    # Convert sets back to comma-separated strings for Pinecone compatibility
    final_docs = list(unique_docs.values())
    for doc in final_docs:
        # Join types: e.g., {"anime", "games"} -> "anime, games"
        doc.metadata["mediatype"] = ", ".join(sorted(doc.metadata["mediatype"]))

    print(f"Deduplication complete. Remaining unique documents: {len(final_docs)}")
    return final_docs




## read and dedup

In [113]:
folders = ('anime', 'manga', 'game', 'light_novel')
# folders = ('anime','manga', 'light_novel')
# folders = ('light_novel',)
all_docs = []
for folder in folders:
    all_docs.extend(load_documents(f"/wiki_data_json/{folder}"))

print("before dedup", len(all_docs))
all_docs = deduplicate_by_filename(all_docs)
print("after dedup", len(all_docs))
print(all_docs[0].page_content)
print(all_docs[0].metadata)


Loading documents from /wiki_data_json/anime...
Loading documents from /wiki_data_json/manga...
Loading documents from /wiki_data_json/game...
Loading documents from /wiki_data_json/light_novel...
before dedup 33783
Deduplication complete. Remaining unique documents: 29908
after dedup 29908
『001/7おや指トム』（ゼロゼロななぶんのいちおやゆびトム）（英語表記：TOM of T.H.U.M.B.）は、アメリカのビデオクラフト社と日本の東映動画による日米合作のテレビアニメである。全24話。
タイトル表記について、『親指トム』や『親ゆびトム』と表記している文献やレコードが多いが、『おや指トム』が正しい。
日本では、アニメ『キングコング』とのセットで放送。NET（現・テレビ朝日）系列局で毎週水曜 19:30 - 20:00 （日本標準時）に放送されていた。番組自体は全26回で、1967年4月5日から同年10月4日まで放送されていたが、ラスト2回を『キングコング』の放送に使うため、本作は同年9月20日放送分をもって終了した。
本作の主人公は、名探偵のヒーローという設定である。対抗する悪の組織として、MAD（またまた悪事同盟）という組織が登場する。

ストーリー
主人公のトムとその助手のジャックが、ちびっ子光線を浴びて小人化してしまった。小人化したことにより、普通の人では解決し得ない難事件を解決していく。

声の出演（日本語吹き替え版）
トム（主人公） - 近石真介
ジャック（トムの助手） - 八代駿
チーフ（トムとジャックの上司） - 熊倉一雄
ジェフ・ブリッジス - 寺田誠
千葉耕市
槐柳二
塚田正昭

主題歌（日本語吹き替え版）
「001/7おや指トム」
作詞・作曲・編曲 - 小林亜星 / 歌 - フォー・シンガーズ / 発売元 - テイチク（現・テイチクエンタテインメント）
この曲が流れるパートの映像は、冒頭のチアガールが持つタイトル部と最後のタイトル表示以外は英語版でも同じであ

In [114]:
types = {doc.metadata['mediatype'] for doc in all_docs}
print(types)
for doc in all_docs:
    if ',' in doc.metadata['mediatype']:
        print(doc.metadata['mediatype'], doc.metadata['source'])
        break

{'game', 'light_novel, manga', 'game, light_novel, manga', 'anime, game', 'manga', 'game, manga', 'anime, light_novel', 'light_novel', 'anime, game, manga', 'anime', 'anime, manga', 'game, light_novel', 'anime, light_novel, manga', 'anime, game, light_novel, manga'}
anime, manga C:\wiki_data_json\anime\07-GHOST.json


In [115]:
split_docs = split_documents(all_docs)


In [116]:
split_docs[5].metadata

{'source': 'C:\\wiki_data_json\\anime\\009-1.json',
 'seq_num': 1,
 'title_ja': '009ノ1',
 'title_en': '009-1',
 'year': 1967,
 'characters': ['ミスナイン', 'ナンバー0', 'ミスタービックラス', '009ノ7', '9グループ'],
 'mediatype': 'anime',
 'author': None,
 'url': 'https://ja.wikipedia.org/wiki/009%E3%83%8E1',
 'id': '001b8d77-5b8a-5c32-a3f0-b3ef3380d875'}

In [117]:
def doc_2_str(document):
    docs_str =  f"作品名: {document.metadata['title_ja']}\n"
    docs_str += f"type: {document.metadata['mediatype']}\n"
    if document.metadata['title_en']:
        docs_str +=  f"英語: {document.metadata['title_en']}\n"
    if document.metadata['author']:
        docs_str +=  f"原作: {document.metadata['author']}\n"
    if document.metadata['year']:
        docs_str +=  f"年: {document.metadata['year']}\n"
    if document.metadata["characters"]:
        docs_str += f"主要キャラ: {", ".join(document.metadata["characters"])}\n"
    docs_str += "---\n"
    docs_str += document.page_content
    return docs_str

corpus = [doc_2_str(doc) for doc in split_docs]

print(corpus[4])

作品名: 009ノ1
type: anime
英語: 009-1
年: 1967
主要キャラ: ミスナイン, ナンバー0, ミスタービックラス, 009ノ7, 9グループ
---
イーストブロック諜報員
ゴリラ男
第1話「頭脳を探せ！」に登場。ゴリラの身体に脳を移植している。クライン博士の頭脳をめぐってクライン博士の助手ナカヤマと取引する。
閣下
第2話「Dr.Xを連行せよ！」に登場。イーストブロックの軍人。スキンヘッドで顔の右側に縫合跡があり、右目に眼帯をしている。Dr.Xの研究を収めたマイクロフィルムのありかを聞き出すため、ミレーヌを拷問する。
リチャード
第3話「超能力者をわがポケットに」に登場。イギリスのチベット遺跡調査隊を全滅させ、生き残りを装ってミレーヌに近づいた。
シャルル・アブナー
第7話に登場。経験の浅い新人スパイ。長髪で片目を隠している。同僚であるクラリスの死にショックを受け、ミレーヌに慰められるうちに愛情が芽生える。ミレーヌを誘い2人で城から脱出しようとするが失敗し騎士ロボットに殺される。
クラリス
第7話に登場。シャルルに好意を持っておりミレーヌに惹かれていくシャルルに嫉妬する。落とし穴に落とされ毒蛇に噛まれて死亡。
ムーアヘッド
第7話に登場。Mr.リヴェンジの招きに応じてリヴェンジ城にやってきたイーストブロックのスパイ達のリーダー。容姿は髭を生やして壮年になった007。020と同様、銃撃戦でMr.リヴェンジの恋人・マリアを撃ってしまった。020と共にMr.リヴェンジに処刑される。
ナターシャ大佐
第9話「魚が出てきた日」に登場。部下であるイワンと共にミレーヌと謎の物質Xの争奪戦を繰り広げた。
イワン
第9話に登場。大柄なナターシャ大佐とは対称的な小柄な男。大佐にこき使われている。
記憶喪失の少女
第11話「Give and Take」に登場。レイプされたショックで海に身を投げてしまいナンバー0とミレーヌに助けられる。ナンバー0を父だと思い込み、記憶が回復するまでナンバー0の家に預けられる。
女性型サイボーグ
第11話に登場。ナンバー0暗殺を命じられたサイボーグ。4本の指先と口からビーム、バストからミサイルを発射する。首・腕・上半身が分離し、それぞれ飛行しながら攻撃することも可能。腰の部分に記憶喪失の少女の脳が収められている。
パロディ
第

In [118]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

# Create the index if it doesn't exist
index_name = "japanese-wiki-index"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="dotproduct", # Best for Voyage/OpenAI
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1" # Or your preferred region
        )
    )

In [119]:
from pinecone_text.sparse import BM25Encoder

bm25 = BM25Encoder(
    language="english", ## set to English because japanese not supported. Turn off english text processing.
    lower_case=False,
    remove_punctuation=True,
    remove_stopwords=False,
    stem=False
)

def mecab_japanese_tokenizer(text: str) -> list[str]:
    """Splits Japanese sentences into individual word tokens."""
    parsed = tagger.parse(text)
    # MeCab output ends with a newline, strip it and split by spaces
    return parsed.strip().split(" ")

bm25.tokenizer = mecab_japanese_tokenizer

bm25.fit(corpus)



100%|██████████| 204022/204022 [01:40<00:00, 2033.65it/s]


In [120]:
# Try generating a clean sparse vector
target_doc = "作品名: ウマ娘 プリティーダービー\n主要キャラ: スペシャルウィーク"
sparse_vector = bm25.encode_documents(target_doc)
print(sparse_vector)


{'indices': [1613295557, 306375727, 1189834286, 3245780369, 2367079478], 'values': [0.7255426753118157, 0.7255426753118157, 0.7255426753118157, 0.7255426753118157, 0.7255426753118157]}


In [133]:
import voyageai
import os
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()

vo = voyageai.Client(api_key=os.environ.get("VOYAGE_API_KEY"))
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index = pc.Index("japanese-wiki-index")

def get_dense_vector(final_text_to_embed):
    dense_response = vo.embed(
        texts=[final_text_to_embed],
        model="voyage-4-lite",
        input_type="document"
    )
    return dense_response.embeddings[0]

def get_sparse_vector(final_text_to_embed):
    return bm25.encode_documents(final_text_to_embed)


def get_payload(doc):
    to_embed = doc_2_str(doc)
    metadata = {}
    for key, val in doc.metadata.items():
        if val is not None:
            if not isinstance(val, str) and not isinstance(val, list):
                metadata[key] = val
            else:
                if len(val) > 0:
                    metadata[key] = val
    metadata["text"] = doc.page_content
    payload = {"id": doc.metadata['id'],
               "values": get_dense_vector(to_embed),
               "sparse_values": get_sparse_vector(to_embed),
               "metadata": metadata
               }
    return payload

def chunk_list(data_list, batch_size):
    """Yield successive batch_size chunks from data_list."""
    for i in range(0, len(data_list), batch_size):
        yield data_list[i : i + batch_size]


# 2. Batch Upload to Pinecone
BATCH_SIZE = 100
print("Starting batch upload to Pinecone...")

# Call the custom chunk_list function
for batch in chunk_list(split_docs, BATCH_SIZE):
    payloads = [get_payload(doc) for doc in batch]
    payloads = [payload for payload in payloads if payload["metadata"]["mediatype"]!= "game"]
    index.upsert(vectors=payloads)
    print(f"Uploaded a batch of {len(batch)} vectors.")



Starting batch upload to Pinecone...
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Uploaded a batch of 100 vectors.
Upload

PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Wed, 27 May 2026 07:51:05 GMT', 'Content-Type': 'application/json', 'Content-Length': '52', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '23', 'x-envoy-upstream-service-time': '23', 'x-pinecone-response-duration-ms': '24', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Invalid request.","details":[]}


In [125]:
len(split_docs)

204022

In [136]:
# ... (Keep your existing clients and helper functions) ...

BATCH_SIZE = 50  # Lowered from 100 to safely stay under the 2MB payload limit


def safe_upsert_individual_docs(batch_data):
    """Fallback function: Processes and uploads documents one-by-one

    to isolate and skip the exact corrupted row causing the 400 error.
    """
    for doc in batch_data:
        try:
            to_embed = doc_2_str(doc)
            dense_response = vo.embed(
                texts=[to_embed], model="voyage-4-lite", input_type="document"
            )

            metadata = doc.metadata.copy()
            metadata["text"] = doc.page_content
            clean_metadata = {k: v for k, v in metadata.items() if v is not None}

            sparse_vector = bm25.encode_documents(to_embed)

            payload = {
                "id": doc.metadata["id"],
                "values": dense_response.embeddings[0],
                "sparse_values": sparse_vector,
                "metadata": clean_metadata,
            }

            index.upsert(vectors=[payload])
        except Exception as individual_error:
            # This line catches and logs the exact bad document, then keeps moving!
            print(
                f"SKIPPING CORRUPTED DOC [ID: {doc.metadata.get('id')}]: {individual_error}"
            )


print("Resuming batch upload with crash protection...")

for batch in chunk_list(split_docs[154500:], BATCH_SIZE):
    try:
        # Standard fast batch processing
        texts_to_embed = [doc_2_str(doc) for doc in batch]

        dense_response = vo.embed(
            texts=texts_to_embed, model="voyage-4-lite", input_type="document"
        )
        dense_vectors = dense_response.embeddings

        payloads = []
        for idx, doc in enumerate(batch):
            to_embed = texts_to_embed[idx]
            metadata = doc.metadata.copy()
            metadata["text"] = doc.page_content
            clean_metadata = {k: v for k, v in metadata.items() if v is not None}

            sparse_vector = bm25.encode_documents(to_embed)

            payloads.append(
                {
                    "id": doc.metadata["id"],
                    "values": dense_vectors[idx],
                    "sparse_values": sparse_vector,
                    "metadata": clean_metadata,
                }
            )

        index.upsert(vectors=payloads)
        print(f"Uploaded a batch of {len(batch)} vectors.")

    except Exception as batch_error:
        print(
            f"⚠️ Batch failed due to data anomaly. Isolating problematic row..."
        )
        # If the batch fails, trigger the fallback to process them individually
        safe_upsert_individual_docs(batch)

print("All rest vectors successfully indexed!")

Resuming batch upload with crash protection...
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 vectors.
Uploaded a batch of 50 ve

In [137]:
bm25.dump("japanese_bm25_model.json")

In [ ]:
204